# Image Recognition



## MNIST dataset

&ensp; &ensp; &ensp; &ensp; So far, we have only used the 60,000 training images of the [MNIST dataset](https://huggingface.co/datasets/ylecun/mnist). But, we have not talked about how this images have been imported and preprocessed to be used in our neural network.

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
import numpy as np

In [ ]:
# load the MNIST dataset
ds = load_dataset("ylecun/mnist")
ds

Based on the dataset structure, we can access the images like this:

In [3]:
first_image = ds['train'][0]['image']
first_image

Please note that the images in the dataset are in PIL (Python Imaging Library) format.

In [4]:
type(first_image)

PIL.PngImagePlugin.PngImageFile

## Data Preprocessing

Usually, the collected data is not immediately ready to be the input of a machine learning model. The steps taken to clean and prepare the data are known as **data preprocessing**. When working with PIL images in deep learning, the typical workflow involves converting them first to NumPy arrays and then, to normalized PyTorch tensors. 

Suppose we have the following 4x4 PIL image:

```{figure} ../images/image-example.png
---
width: 150px
name: image-example
---
4x4 PIL image
```

When we convert this image to a NumPy array, each pixel value ranges from 0 (black) to 255 (white), with intermediate values representing varying shades of gray.

```python
np.array([[  0,  42,  85, 127],
          [ 42,  85, 127, 170],
          [ 85, 127, 170, 213],
          [127, 170, 213, 255]])
```

Neural networks perform better when when we **normalize** the input values so that they fall within the range [0, 1]. To do this, we divide each pixel value by 255, so values range from 0 (black) to 1 (white).

```python
tensor([[0.0000, 0.1647, 0.3333, 0.4980],
        [0.1647, 0.3333, 0.4980, 0.6667],
        [0.3333, 0.4980, 0.6667, 0.8353],
        [0.4980, 0.6667, 0.8353, 1.0000]])
```

We will create the function `preprocess_data` to apply these transformations to all the images in the train and test datasets.

In [90]:
# convert PIL image to normalized PyToch tensor
def image_to_tensor(image):
    return torch.tensor(np.array(image)) / 255.0


def preprocess_data(split):
    
    x = []  # list to store image tensors
    y = []  # list to store labels

    for example in split:
        x.append(image_to_tensor(example['image']))
        y.append(example['label'])
    
    return torch.stack(x), torch.tensor(y)


# preprocess the train and test datasets
train_x, train_y = preprocess_data(ds['train'])
test_x,  test_y  = preprocess_data(ds['test'])

## Train, Test, and Validation Splits

When working with neural networks, it is common practice to divide the dataset into three splits:

- The **train split** (80% of the dataset) is used to optimize the model parameters during training.
- The **validation split** (10% of the dataset) is used to help us adjust the hyperparameters of the model, such as the size of the hidden layer, embedding size, and regularization strength.
- The **test split** (10% of the dataset) is used to evaluate the final model's performance.

```{important}
The test split should be used only after the model is fully optimized, as its goal is to evaluate the final model's performance. Using the test split to help us adjust the hyperparameters of the model would result in overfitting.

In [91]:
# Split test data into validation and test data
val_x = test_x[:5000]
val_y = test_y[:5000]
test_x = test_x[5000:]
test_y = test_y[5000:]

In [92]:
print(f"{'Splits':<15}{'Images':<20}{'Labels'}")
print("-" * 50)
print(f"{'Train':<15}{str(tuple(train_x.shape)):<20}{str(list(train_y.shape))}")
print(f"{'Validation':<15}{str(tuple(val_x.shape)):<20}{str(list(val_y.shape))}")
print(f"{'Test':<15}{str(tuple(test_x.shape)):<20}{str(list(test_y.shape))}")

Splits         Images              Labels
--------------------------------------------------
Train          (60000, 28, 28)     [60000]
Validation     (5000, 28, 28)      [5000]
Test           (5000, 28, 28)      [5000]


In [98]:
train_x = train_x.view(-1, 784)
val_x = val_x.view(-1, 784)
test_x = test_x.view(-1, 784)

## Flatten

In [99]:
print(f"{'Splits':<15}{'Images':<20}{'Labels'}")
print("-" * 50)
print(f"{'Train':<15}{str(tuple(train_x.shape)):<20}{str(list(train_y.shape))}")
print(f"{'Validation':<15}{str(tuple(val_x.shape)):<20}{str(list(val_y.shape))}")
print(f"{'Test':<15}{str(tuple(test_x.shape)):<20}{str(list(test_y.shape))}")

Splits         Images              Labels
--------------------------------------------------
Train          (60000, 784)        [60000]
Validation     (5000, 784)         [5000]
Test           (5000, 784)         [5000]


## nn.Module, nn.Sequential and nn.Layers

In [104]:
from torch.nn import Module, Sequential, Linear, BatchNorm1d, Tanh

class PrevNN(Module):

    def __init__(self, n_hidden=100):
        super().__init__()
        
        self.layers = Sequential(
            Linear(     784, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
            Linear(n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
            Linear(n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
            Linear(n_hidden, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
            Linear(n_hidden,       10, bias=False)
        )
    
    def forward(self, x):
        return self.layers(x)



def get_batch(batch_size):
    ix = torch.randint(0, 60000, (batch_size,))
    return train_x[ix], train_y[ix]


@torch.no_grad()
def calculate_loss(step):

    model.eval() # put the model in evaluation mode

    logits = model(train_x.view(-1, 784))
    train_loss = F.cross_entropy(logits, train_y)

    logits = model(val_x.view(-1, 784))
    val_loss = F.cross_entropy(logits, val_y)

    model.train() # put the model back in train mode
    
    print(f"Step: {step:3d}/{steps}     Train Loss: {train_loss:.4f}     Val Loss: {val_loss:.4f}")

In [105]:
# hyperparameters
steps = 200        # train iterations
lr = 0.001           # learning rate
batch_size = 32
eval_interval = 10  # every how many iterations calculate the loss

loss_graph = []

# initialize the model
model = PrevNN(n_hidden=100)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for step in range(steps):

    if step % eval_interval == 0 or step == steps - 1:
        calculate_loss(step)
    
    # sample a batch of the dataa
    Xb, Yb = get_batch(batch_size)

    # forward pass
    logits = model(Xb)

    # calculate loss
    loss = F.cross_entropy(logits, Yb)

    # backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # update parameters
    optimizer.step()

Step:   0/200     Train Loss: 2.2998     Val Loss: 2.3017
Step:  10/200     Train Loss: 1.8046     Val Loss: 1.8387
Step:  20/200     Train Loss: 1.1070     Val Loss: 1.1446
Step:  30/200     Train Loss: 0.7830     Val Loss: 0.8461
Step:  40/200     Train Loss: 0.6658     Val Loss: 0.7329
Step:  50/200     Train Loss: 0.5699     Val Loss: 0.6440
Step:  60/200     Train Loss: 0.5145     Val Loss: 0.5920
Step:  70/200     Train Loss: 0.4878     Val Loss: 0.5629
Step:  80/200     Train Loss: 0.4582     Val Loss: 0.5261
Step:  90/200     Train Loss: 0.4354     Val Loss: 0.5016
Step: 100/200     Train Loss: 0.4355     Val Loss: 0.5033
Step: 110/200     Train Loss: 0.4459     Val Loss: 0.5364
Step: 120/200     Train Loss: 0.3985     Val Loss: 0.4934
Step: 130/200     Train Loss: 0.3763     Val Loss: 0.4634
Step: 140/200     Train Loss: 0.3889     Val Loss: 0.4822
Step: 150/200     Train Loss: 0.3714     Val Loss: 0.4540
Step: 160/200     Train Loss: 0.3590     Val Loss: 0.4220
Step: 170/200 

## Overfitting

## Confusion Matrix

In [106]:
logits.shape

torch.Size([32, 10])

In [107]:
logits

tensor([[-1.1193,  5.4148, -0.2775, -1.6869, -0.0273, -1.3072, -0.5260,  0.5073,
          0.3646, -1.5853],
        [-2.4561,  4.6136, -1.1070,  0.4441, -0.9890, -0.4395, -1.6039, -1.2998,
          2.5566, -0.7568],
        [ 1.2746, -1.3047,  4.7556,  1.2086, -2.2024,  1.0423, -0.8071,  0.3263,
         -0.4378, -3.7468],
        [-0.4872, -1.7870, -1.0523, -1.3695,  0.6247, -0.8319, -2.3753,  0.4271,
          0.1690,  4.9954],
        [-0.6980, -0.7078,  0.4569,  6.1222, -1.7360,  0.2125, -2.3767,  0.1138,
          0.1765, -1.3904],
        [ 1.3695, -1.8139, -1.2227, -3.2725,  4.0267,  1.3369,  0.7050, -0.7680,
         -0.1585,  0.2266],
        [-1.3365, -0.1346, -0.3530, -0.7961, -0.1136, -1.1548, -0.8300,  5.1387,
         -2.0105,  2.1827],
        [-0.0631, -0.8508, -0.9687, -0.6707, -0.4089, -0.9874, -1.5830,  5.4267,
         -2.0847,  2.0747],
        [-1.9058,  6.2446, -0.2088, -0.8048, -1.0710, -1.2578, -0.2935, -0.2083,
          0.4249, -1.2028],
        [-2.4451,  

In [120]:
pred = torch.argmax(logits, dim=1)
pred

tensor([1, 1, 2, 9, 3, 4, 7, 7, 1, 1, 1, 6, 7, 2, 9, 9, 6, 3, 2, 2, 6, 6, 2, 3,
        8, 6, 9, 4, 7, 4, 3, 0])

In [117]:
Yb

tensor([1, 1, 2, 7, 3, 4, 7, 7, 1, 1, 1, 6, 7, 2, 9, 7, 6, 3, 2, 2, 6, 6, 2, 3,
        8, 6, 4, 4, 7, 4, 3, 2])

In [119]:
conf_matrix = torch.zeros(10, 10, dtype=torch.int32)

for t, p in zip(Yb, pred):
    conf_matrix[t, p] += 1

conf_matrix

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 5, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 0, 5, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 4, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 3, 0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 5, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 4, 0, 2],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]], dtype=torch.int32)